<a href="https://colab.research.google.com/github/srikalyan123/FineTunningLLM/blob/main/Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# pip install transformers datasets

In [2]:
from transformers import AutoTokenizer
from datasets import Dataset
from sklearn.model_selection import train_test_split

# 1. Sample Data
# In a real scenario, you would load your own dataset here.
# For demonstration, we'll create a simple list of sentences.
sentences = [
    "Hello, this is a sample sentence for tokenization.",
    "Hugging Face provides excellent tools for NLP tasks.",
    "Tokenization is the process of breaking text into smaller units.",
    "We will split this data into train, validation, and test sets.",
    "Large Language Models (LLMs) rely heavily on efficient tokenization.",
    "The 'datasets' library makes it easy to handle data for ML.",
    "Sklearn's train_test_split is useful for initial data partitioning."
]

# 2. Load a pre-trained tokenizer
# You can choose any tokenizer compatible with your LLM (e.g., 'bert-base-uncased', 'gpt2', 'roberta-base')
# For this example, let's use a common one.
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 3. Tokenize the sentences
# The `padding='max_length'` and `truncation=True` are important for batch processing.
# `return_tensors='pt'` returns PyTorch tensors, which are commonly used in Transformers.
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Convert the list of sentences into a Dataset object
# The Dataset library expects a dictionary-like structure.
data_dict = {"text": sentences}
dataset = Dataset.from_dict(data_dict)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

print("Original dataset features:", tokenized_datasets.features)
print("Example of tokenized input_ids:", tokenized_datasets[0]["input_ids"])
print("Example of tokenized attention_mask:", tokenized_datasets[0]["attention_mask"])

# 4. Split the dataset into train, validate, test
# First, split into training and a temporary set (validation + test)
train_val_test_split = tokenized_datasets.train_test_split(test_size=0.3, seed=42) # 70% train, 30% for val/test

train_dataset = train_val_test_split['train']

# Now, split the remaining 30% into validation and test sets (50/50 split of the 30%)
# This results in 70% train, 15% validation, 15% test
val_test_split = train_val_test_split['test'].train_test_split(test_size=0.5, seed=42)

validation_dataset = val_test_split['train']
test_dataset = val_test_split['test']

print(f"\nDataset sizes:")
print(f"Train set size: {len(train_dataset)}")
print(f"Validation set size: {len(validation_dataset)}")
print(f"Test set size: {len(test_dataset)}")

print("\nFirst entry of training dataset:")
print(train_dataset[0])

/Users/srikalyanchakravarthimarri/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/srikalyanchakravarthimarri/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 7/7 [00:00<00:00, 1359.71 examples/s]

Original dataset features: {'text': Value('string'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}
Example of tokenized input_ids: [101, 7592, 1010, 2023, 2003, 1037, 7099, 6251, 2005, 19204, 3989, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Example of tokenized attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

### Loading an external dataset for tokenization and splitting

For a more realistic scenario and to meet the requirement of at least 1000 sentences, I'll use the 'emotion' dataset from the Hugging Face `datasets` library. This dataset is suitable for tasks related to understanding the emotional tone of text, which aligns with your goal of developing an LLM that can understand conversation sensitivity and customer behavior.

In [9]:
from datasets import load_dataset

# Load the 'emotion' dataset
# This dataset is a good example for sentiment/emotion analysis and has over 16,000 sentences.
dataset = load_dataset("dair-ai/emotion")

print("Loaded dataset structure:", dataset)

# Access the 'train' split of the dataset for demonstration
# The 'emotion' dataset already has train, validation, and test splits.
# However, we will re-split it according to your specified ratios (50/25/25).
Train_data = dataset['train']['text']
Validatem_data = dataset['validation']['text']
Test_data = dataset['test']['text']

print(f"\nTotal sentences in the 'train' split of the emotion dataset: {len(Train_data)}")
print(f"Total sentences in the 'validation' split of the emotion dataset: {len(Validatem_data)}")
print(f"Total sentences in the 'test' split of the emotion dataset: {len(Test_data)}")
print("\nFirst entry of the 'train' split:")
print(Train_data[0])

ModuleNotFoundError: No module named 'datasets'

Now, let's tokenize this loaded dataset and apply the custom split ratios (50% train, 25% validation, 25% test).

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Re-using the tokenizer defined previously
# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# tokenize_function = lambda examples: tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Convert the full_text_data list into a Dataset object
data_dict = {"text": Train_data}
Train_dataset = Dataset.from_dict(data_dict)

# Tokenize the full dataset
tokenized_train_dataset = Train_dataset.map(tokenize_function, batched=True)

print("\nTokenized dataset features:", tokenized_train_dataset.features)
print("Example of tokenized input_ids:", tokenized_train_dataset[0]["input_ids"])
print("Example of tokenized token_type_ids:", tokenized_train_dataset[0]["token_type_ids"])
print("Example of tokenized attention_mask:", tokenized_train_dataset[0]["attention_mask"])


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]


Tokenized dataset features: {'text': Value('string'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}
Example of tokenized input_ids: [101, 1045, 2134, 2102, 2514, 26608, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Example of tokenized token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import Dataset
# We will use 'evaluate' library for metrics, not 'datasets.load_metric'
# import numpy as np
# from datasets import load_metric

# --- Re-tokenize and prepare all datasets with labels ---

# For training, we need the labels in the tokenized_train_dataset.
# The original `tokenized_train_dataset` (from cell 17334bbb) did not explicitly include labels.
# Let's recreate it ensuring labels are present.

train_labels = dataset['train']['label'] # 'dataset' is from cell '0da13b27'
train_data_with_labels_dict = {"text": Train_data, "label": train_labels}
train_raw_dataset = Dataset.from_dict(train_data_with_labels_dict)
tokenized_train_dataset = train_raw_dataset.map(tokenize_function, batched=True)

# Tokenize the validation dataset
val_labels = dataset['validation']['label']
val_data_dict = {"text": Validatem_data, "label": val_labels}
validation_raw_dataset = Dataset.from_dict(val_data_dict)
tokenized_validation_dataset = validation_raw_dataset.map(tokenize_function, batched=True)

# Tokenize the test dataset
test_labels = dataset['test']['label']
test_data_dict = {"text": Test_data, "label": test_labels}
test_raw_dataset = Dataset.from_dict(test_data_dict)
tokenized_test_dataset = test_raw_dataset.map(tokenize_function, batched=True)

print("Tokenized train dataset features:", tokenized_train_dataset.features)
print("Tokenized validation dataset features:", tokenized_validation_dataset.features)
print("Tokenized test dataset features:", tokenized_test_dataset.features)

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenized train dataset features: {'text': Value('string'), 'label': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}
Tokenized validation dataset features: {'text': Value('string'), 'label': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}
Tokenized test dataset features: {'text': Value('string'), 'label': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}


Now that all datasets are properly tokenized and include labels, we can proceed with model loading, training, and evaluation.

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [ ]:
# 1. Get the number of labels and their mappings from the dataset
num_labels = dataset['train'].features['label'].num_classes
label_names = dataset['train'].features['label'].names
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

print(f"Number of labels: {num_labels}")
print(f"Label to ID mapping: {label2id}")
print(f"ID to Label mapping: {id2label}")

# 2. Load a pre-trained model for sequence classification
# We'll use a BERT-based model compatible with the tokenizer.
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# 3. Define training arguments
training_args = TrainingArguments(
    output_dir="./results",          # Output directory for model predictions and checkpoints
    eval_strategy="epoch",         # Corrected: Use eval_strategy instead of evaluation_strategy
    learning_rate=2e-5,              # Learning rate
    per_device_train_batch_size=16,  # Batch size for training
    per_device_eval_batch_size=16,   # Batch size for evaluation
    num_train_epochs=3,              # Total number of training epochs
    weight_decay=0.01,               # Strength of weight decay
    logging_dir="./logs",            # Directory for storing logs
    logging_steps=100,               # Log every X updates steps
    save_strategy="epoch",           # Save checkpoint every epoch
    load_best_model_at_end=True,     # Load the best model at the end of training
    metric_for_best_model="accuracy",# Metric to use for best model selection
    report_to="none"                 # Disable integration with other tools like wandb
)

# 4. Create a Data Collator
# A DataCollator pads your inputs to the longest sequence in a batch.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 5. Define a compute_metrics function for evaluation
import numpy as np
# Import load_metric from the 'evaluate' library
import evaluate

def compute_metrics(eval_pred):
    load_accuracy = evaluate.load("accuracy")
    load_f1 = evaluate.load("f1")

    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = load_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = load_f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": accuracy, "f1": f1}

# 6. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset,
    # Removed tokenizer=tokenizer as it's not an accepted argument for Trainer directly
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 7. Train the model
print("\nStarting model training...")
trainer.train()
print("Model training complete!")

# You can also evaluate the model on the test set after training
print("\nEvaluating model on the test set...")
eval_results = trainer.evaluate(tokenized_test_dataset)
print(f"Test Set Evaluation Results: {eval_results}")

Number of labels: 6
Label to ID mapping: {'sadness': 0, 'joy': 1, 'love': 2, 'anger': 3, 'fear': 4, 'surprise': 5}
ID to Label mapping: {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `loggin


Starting model training...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Model training complete!

Evaluating model on the test set...


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.081310,0.181623,3,0.929500,0.928933


Test Set Evaluation Results: {'eval_loss': 0.18162313103675842, 'eval_accuracy': 0.9295, 'eval_f1': 0.9289333708724506}


### Understanding Training and Validation Loss

During the training process, a Transformer model learns to minimize a loss function. The loss is a measure of how well the model is performing on a given task.

*   **Training Loss**: This is the loss calculated on the data the model is actively learning from (the `train_dataset`). A decreasing training loss indicates that the model is learning from the data.

*   **Validation Loss**: This is the loss calculated on a separate dataset (`eval_dataset`) that the model has not seen during training. It's used to monitor the model's performance on unseen data and helps detect overfitting. If the training loss continues to decrease but the validation loss starts to increase, it's a strong sign of overfitting.

*   **Test Loss**: This is the loss calculated on the `test_dataset` after the model has completed training. It gives an unbiased estimate of the model's performance on completely new, unseen data.

**Interpreting the Results:**

*   **Both training and validation loss decrease**: This is a good sign that your model is learning and generalizing well.
*   **Training loss decreases, validation loss increases**: The model is overfitting to the training data. You might need to adjust hyperparameters, add regularization, or use more data.
*   **Both losses are high**: The model might be underfitting, meaning it's not complex enough or hasn't trained long enough. Consider increasing model capacity or training for more epochs.
*   **Accuracy/F1 Score**: These metrics provide a more interpretable measure of classification performance. Higher values generally indicate a better model.

In the output of the previous cell (`4cdb93d2`), you can observe the training loss reported periodically during the training epochs, and the validation loss and metrics (like accuracy and F1-score) reported at the end of each epoch (`eval_strategy="epoch"`). The final `Test Set Evaluation Results` show the model's performance on the unseen test dataset.

### How to Test the Fine-tuned LLM

After fine-tuning, you can test your LLM by passing new text inputs to it and observing its predictions. Since this model was fine-tuned for sequence classification (emotion analysis), we'll expect it to classify the emotion of a given sentence.

In [ ]:
# Function to make a prediction on a new sentence
def predict_emotion(text):
    # Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt", padding="max_length", truncation=True, max_length=128)

    # Move inputs to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Make a prediction
    with torch.no_grad(): # Disable gradient calculations for inference
        outputs = model(**inputs)

    # Get the logits
    logits = outputs.logits

    # Get the predicted class ID
    predicted_class_id = logits.argmax().item()

    # Map the class ID back to the label name
    predicted_label = id2label[predicted_class_id]
    return predicted_label

# Example usage:
import torch

# Assuming `model` and `tokenizer` are loaded and `id2label` is defined from previous cells.

sentence1 = "I am so happy today, this is wonderful!"
sentence2 = "I feel really sad and lonely."
sentence3 = "I am so surprised by this news."
sentence4 = "I am furious about what happened."

print(f"Sentence: '{sentence1}' -> Predicted Emotion: {predict_emotion(sentence1)}")
print(f"Sentence: '{sentence2}' -> Predicted Emotion: {predict_emotion(sentence2)}")
print(f"Sentence: '{sentence3}' -> Predicted Emotion: {predict_emotion(sentence3)}")
print(f"Sentence: '{sentence4}' -> Predicted Emotion: {predict_emotion(sentence4)}")


Sentence: 'I am so happy today, this is wonderful!' -> Predicted Emotion: joy
Sentence: 'I feel really sad and lonely.' -> Predicted Emotion: sadness
Sentence: 'I am so surprised by this news.' -> Predicted Emotion: surprise
Sentence: 'I am furious about what happened.' -> Predicted Emotion: anger


## Uploading the Model to Hugging Face Hub

To share your fine-tuned model and tokenizer with the community (or just save it privately), you can upload it to the Hugging Face Hub. This process involves a few steps:

1.  **Authenticate**: You need to log in to the Hugging Face Hub from your Colab environment.
2.  **Push to Hub**: Use the `push_to_hub()` method provided by the `Trainer` object to upload the model, tokenizer, and automatically generate a basic model card.

In [ ]:
# Log in to Hugging Face Hub
# You will be prompted to enter your Hugging Face token.
# You can find your token at https://huggingface.co/settings/tokens
from huggingface_hub import notebook_login

notebook_login()

Once authenticated, you can push your model and tokenizer. You need to decide on a repository name. It's usually in the format `your_username/your_model_name`.

In [ ]:
# Define your Hugging Face repository ID
# Replace 'your_username' with your Hugging Face username
# Replace 'bert-emotion-classifier' with your desired model name
repo_id = "srikalyanmarri/bert-emotion-classifier"

# Push the model and tokenizer to the Hugging Face Hub
# This will also generate a basic model card based on your training arguments.
print(f"Pushing model to Hugging Face Hub under repository: {repo_id}")
trainer.push_to_hub(repo_id)
print("Model successfully pushed to Hugging Face Hub!")

Pushing model to Hugging Face Hub under repository: srikalyanmarri/bert-emotion-classifier


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...results/model.safetensors:   0%|          | 14.2kB /  438MB            

  ...results/training_args.bin:   2%|1         |  80.0B / 5.20kB            

Model successfully pushed to Hugging Face Hub!


### Model Card

When you use `trainer.push_to_hub()`, a basic model card (`README.md`) is automatically generated in your repository on the Hugging Face Hub. This model card includes information about the model, training parameters, and evaluation results.

After the upload is complete, you can visit your repository on `huggingface.co` (e.g., `https://huggingface.co/your_username/bert-emotion-classifier`) to view and further edit the model card to add more details, usage examples, ethical considerations, and other relevant information.